## AuxTel Tracking tests

Attempt to repeat tracking issues in the day with closed dome.

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u

/opt/lsst/software/stack/conda/miniconda3-py38_4.9.2/envs/lsst-scipipe-0.4.1/lib/python3.8/site-packages/jose/backends/cryptography_backend.py:23: CryptographyDeprecationWarning: int_from_bytes is deprecated, use int.from_bytes instead
  from cryptography.utils import int_from_bytes, int_to_bytes


In [2]:
# Set Cerro Pachon location
location = EarthLocation.from_geodetic(lon=-70.747698*u.deg,
                                       lat=-30.244728*u.deg,
                                       height=2663.0*u.m)

In [3]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [4]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [7]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.16 sec
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, algorithm)
Read 27 history items for RemoteEvent(ATDomeTrajectory, 0, appliedSettingsMatchStart)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, authList)
Read 100 history items for RemoteEvent(ATDomeTrajectory, 0, heartbeat)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, logLevel)
Read 100 history items for RemoteEvent(ATDomeTrajectory, 0, logMessage)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, settingVersions)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, settingsApplied)
Read 1 history

[[None, None, None, None, None, None, None], [None, None, None, None]]

timeAndDate DDS read queue is full (100 elements); data may be lost


In [8]:
# enable components if required
# Failed
await atcs.enable()
await latiss.enable()

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '                                                                                                                               ', 'atptg': '', 'ataos': 'current', 'atpneumatics': '                                                                                                                               ', 'athexapod': 'summit', 'atdome': 'test', 'atdometrajectory': ''}
[atmcs]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atptg]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[ataos]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atpneumatics]::[<State.STANDBY: 5>, <State.DISABLED: 

RuntimeError: Failed to transition ['atcamera', 'atarchiver'] to <State.ENABLED: 2>.

In [10]:
await latiss.rem.atcamera.cmd_start.set_start(timeout=30)

In [11]:
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.ENABLED)

[<State.DISABLED: 1>, <State.ENABLED: 2>]

In [12]:
await latiss.rem.atarchiver.cmd_start.set_start(timeout=30)

In [13]:
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.ENABLED)

[<State.DISABLED: 1>, <State.ENABLED: 2>]

In [ ]:
# Everything now enabled 11:45 AM

In [14]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [15]:
# turn on ATAOS corrections just to make sure the mirror is under air
tmp = await atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=False)

In [16]:
# Ensure we're using Nasmyth 2
await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

In [17]:
# point telescope to desired starting position
start_az=90
start_el=80
start_rot_pa=0
await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)

Sending command
Stop tracking.
Unknown tracking state: 10
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = -000.002 deg; delta Az= +089.999 deg; delta N1 = +000.000 deg; delta N2 = -000.156 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +086.798 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +080.801 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +074.800 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +070.801 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +064.802 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -000.000 deg; del

In [21]:
# get RA/DEC of current telescope Alt/Az position
az = Angle(start_az, unit=u.deg)
el = Angle(start_el, unit=u.deg)
print(f'orig az {az} and el {el}')
time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
# This should be TAI and not UTC... so will be 37s off system clock seconds ??
curr_time_atptg = Time(time_data.mjd, format="mjd")
coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
coord_frame_radec = ICRS()
coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
ra_dec = coord_azel.transform_to(coord_frame_radec)
print('Current Position is: \n {}'.format(coord_azel))
print('Current Position is: \n {}'.format(ra_dec))


orig az 90.0 deg and el 80.0 deg
Current Position is: 
 <AltAz Coordinate (obstime=59276.619159139074, location=(1819093.56876225, -5208411.6827961, -3195180.61110659) m, pressure=0.0 hPa, temperature=0.0 deg_C, relative_humidity=0.0, obswl=1.0 micron): (az, alt) in deg
    (90., 80.)>
Current Position is: 
 <ICRS Coordinate: (ra, dec) in deg
    (324.98476612, -29.83202021)>
target python read queue is filling: 18 of 100 elements


In [22]:
# Slew to starting position and take an image to check headers and tracking
await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0,
                      slew_timeout=240., stop_before_slew=False, wait_settle=True)


await asyncio.sleep(2)
# take a 60s dark
await latiss.take_darks(60.0, 1)


Auto sky angle: 0.0 deg
Sending command
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = -000.046 deg; delta Az= -000.019 deg; delta N1 = +000.000 deg; delta N2 = -004.212 deg 
[Telescope] delta Alt = +000.001 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -001.694 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = +000.002 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -000.002 deg 
Axes in position.
Generating group_id
DARK 0001 - 0001


array([2021030300001])

logMessage DDS read queue is filling: 10 of 100 elements
logMessage DDS read queue is filling: 13 of 100 elements
logMessage DDS read queue is filling: 10 of 100 elements


In [23]:
# now loop from elevation of +80 down to +20 in 5 degree increments
# Should be images 2 - 14
for i in range(13):
    start_el = 80 - i * 5
    # point telescope to desired starting position
    await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
    # get RA/DEC of current telescope Alt/Az position
    az = Angle(start_az, unit=u.deg)
    el = Angle(start_el, unit=u.deg)
    print(f'orig az {az} and el {el}')
    time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
    # This should be TAI and not UTC... so will be 37s off system clock seconds ??
    curr_time_atptg = Time(time_data.mjd, format="mjd")
    coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
    coord_frame_radec = ICRS()
    coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
    ra_dec = coord_azel.transform_to(coord_frame_radec)
    print('Current Position is: \n {}'.format(coord_azel))
    print('Current Position is: \n {}'.format(ra_dec))
    # Slew to starting position and take an image to check headers and tracking
    await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0,
                          slew_timeout=240., stop_before_slew=False, wait_settle=True)
    await asyncio.sleep(2)
    # take a 60s dark
    await latiss.take_darks(60.0, 1)
    


Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -005.706 deg; delta Az= +005.518 deg; delta N1 = +000.000 deg; delta N2 = -003.579 deg 
[Telescope] delta Alt = -005.685 deg; delta Az= +003.108 deg; delta N1 = +000.000 deg; delta N2 = -001.287 deg 
[Telescope] delta Alt = -005.302 deg; delta Az= +000.004 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -004.308 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -002.886 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -001.460 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -000.442

AckError: msg='Command failed', ackcmd=(ackcmd private_seqNum=1556608029, ack=<SalRetCode.CMD_FAILED: -302>, error=6611, result='Rejected : elevation out of range')

logMessage DDS read queue is filling: 12 of 100 elements


In [24]:
# That took about 25 minutes.  So repeat 2 more times
# now loop from elevation of +80 down to +20 in 5 degree increments
# Should be images 15 - 41
for repeat in range(5):
    for i in range(13):
        start_el = 80 - i * 5
        # point telescope to desired starting position
        await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
        # get RA/DEC of current telescope Alt/Az position
        az = Angle(start_az, unit=u.deg)
        el = Angle(start_el, unit=u.deg)
        print(f'orig az {az} and el {el}')
        time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
        # This should be TAI and not UTC... so will be 37s off system clock seconds ??
        curr_time_atptg = Time(time_data.mjd, format="mjd")
        coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
        coord_frame_radec = ICRS()
        coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
        ra_dec = coord_azel.transform_to(coord_frame_radec)
        print('Current Position is: \n {}'.format(coord_azel))
        print('Current Position is: \n {}'.format(ra_dec))
        # Slew to starting position and take an image to check headers and tracking
        await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0,
                              slew_timeout=240., stop_before_slew=False, wait_settle=True)
        await asyncio.sleep(2)
        # take a 60s dark
        await latiss.take_darks(60.0, 1)
    


Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
logMessage DDS read queue is filling: 13 of 100 elements
[Telescope] delta Alt = +065.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +063.008 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +057.013 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +051.014 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +045.012 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +041.013 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; 

CancelledError: 

In [25]:
# Repeat 3 more times at 3 different azimuths
# now loop from elevation of +80 down to +20 in 5 degree increments
# Should be images 42-
for repeat in range(3):
    start_az = 180 - 30 * repeat
    for i in range(13):
        start_el = 80 - i * 5
        # point telescope to desired starting position
        await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
        # get RA/DEC of current telescope Alt/Az position
        az = Angle(start_az, unit=u.deg)
        el = Angle(start_el, unit=u.deg)
        print(f'orig az {az} and el {el}')
        time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
        # This should be TAI and not UTC... so will be 37s off system clock seconds ??
        curr_time_atptg = Time(time_data.mjd, format="mjd")
        coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
        coord_frame_radec = ICRS()
        coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
        ra_dec = coord_azel.transform_to(coord_frame_radec)
        print('Current Position is: \n {}'.format(coord_azel))
        print('Current Position is: \n {}'.format(ra_dec))
        # Slew to starting position and take an image to check headers and tracking
        await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0,
                              slew_timeout=240., stop_before_slew=False, wait_settle=True)
        await asyncio.sleep(2)
        # take a 60s dark
        await latiss.take_darks(60.0, 1)
    


Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -001.018 deg; delta Az= +090.686 deg; delta N1 = -000.000 deg; delta N2 = +003.207 deg 
[Telescope] delta Alt = -000.781 deg; delta Az= +088.974 deg; delta N1 = -000.000 deg; delta N2 = +001.684 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +082.991 deg; delta N1 = -000.000 deg; delta N2 = +000.015 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +078.992 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +072.991 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +068.992 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000

In [26]:
# For shutdown of system
await atcs.stop_tracking()

Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.


In [27]:
# turn off corrections
tmp = await atcs.rem.ataos.cmd_disableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [28]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [29]:
# Putting everything back in standby.
# Failed as it usually does
await atcs.shutdown()

Disabling ATAOS corrections
Disabling ATAOS corrections.
Closing M1 cover vent gates.
Cover state <MirrorCoverState.CLOSED: 6>
M1 cover already closed.
M1 vent state <VentsPosition.CLOSED: 1>
M1 vents already closed.
Close dome.
ATDome Shutter Door is already closed. Ignoring.
Slew dome to Park position.
Sending ATDomeTrajectory to DISABED state. Component will be left in DISABLEDstate or else it may send the ATDome back to alignment with the telescope.
process as completed...
atdometrajectory: <State.DISABLED: 1>
[Dome] delta Az = +167.080 deg
[Dome] delta Az = +166.980 deg
[Dome] delta Az = +166.630 deg
[Dome] delta Az = +166.040 deg
[Dome] delta Az = +165.220 deg
[Dome] delta Az = +164.170 deg
[Dome] delta Az = +162.890 deg
[Dome] delta Az = +161.360 deg
[Dome] delta Az = +159.620 deg
[Dome] delta Az = +157.640 deg
[Dome] delta Az = +155.410 deg
[Dome] delta Az = +152.950 deg
[Dome] delta Az = +150.260 deg
[Dome] delta Az = +147.370 deg
[Dome] delta Az = +144.250 deg
[Dome] delta Az

RuntimeError: Failed to transition ['atdome'] to <State.STANDBY: 5>.

In [30]:
await atcs.rem.atdome.cmd_start.set_start(settingsToApply="test", timeout=30)

In [31]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

[<State.DISABLED: 1>, <State.STANDBY: 5>]

In [32]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
# Everything back in standby